In [0]:
dbutils.widgets.removeAll()

In [0]:
# File Parameters
dbutils.widgets.text("ClientID","","") 
dbutils.widgets.text("FileID","","") 
dbutils.widgets.text("FileLayoutID","","") 
dbutils.widgets.text("FileLayoutDescription","","") 
dbutils.widgets.text("ColumnDelimiter","","") 
dbutils.widgets.text("HasHeader","","") 
dbutils.widgets.text("IgnoreHeader","","") 
dbutils.widgets.text("TextQualifier","","") 

# File to be Processed
dbutils.widgets.text("FullFileName","","")  

# Schema File
dbutils.widgets.text("SchemaFile","","")  

# Processed File 
dbutils.widgets.text("ProcessedPath","","") 

In [0]:
ClientId = dbutils.widgets.get("ClientID")
FileId = dbutils.widgets.get("FileID")
FileLayoutId = dbutils.widgets.get("FileLayoutID")
FileLayoutDescription = dbutils.widgets.get("FileLayoutDescription")
ColumnDelimiter = dbutils.widgets.get("ColumnDelimiter")
HasHeader = dbutils.widgets.get("HasHeader")
IgnoreHeader = dbutils.widgets.get("IgnoreHeader")
TextQualifier = dbutils.widgets.get("TextQualifier")
FullFileName = dbutils.widgets.get("FullFileName")
SchemaFile = dbutils.widgets.get("SchemaFile")
ProcessedPath = dbutils.widgets.get("ProcessedPath")


In [0]:
%run "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/Shared/CommonMethods/ABC/FileHandling"

In [0]:
%run "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/Shared/CommonMethods/ABC/SynJSONCreatorClass"

In [0]:
from pyspark.sql.types import StructType
from pyspark.sql.functions import lit, to_timestamp, current_timestamp

ErrorMessage = ""
doubleQuote = '"'

# get job id
try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    currentJobId = ctx.tags().get("jobId").getOrElse(lambda: "Undefined")
except Exception:
    # serverless fallback - ctx.tags() is not whitelisted on serverless
    currentJobId = "Undefined"

In [0]:
def process_move_file(ClientId, FileId, FileLayoutId, FileLayoutDescription,
                      ColumnDelimiter, HasHeader, IgnoreHeader, textQualifier,
                      FullFileName, SchemaFile, ProcessedPath):
    rJSON = synJSONCreator()
    ErrorMessage = ""
    doubleQuote = '"'

        # Get job ID - Serverless compatible
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        currentJobId = ctx.tags().get("jobId").getOrElse(lambda: "Undefined")
    except Exception:
        currentJobId = "Undefined"
    
    rJSON.addBraceStart()
    rJSON.addNewEntry("CurrentJobId", currentJobId)

    dfFile = spark.createDataFrame([], StructType([]))

    # 1. Read the file based on header rules (either keeping or ignoring the header row)
    try:
        if IgnoreHeader == "False":
            dfFile = delimitedFile(FullFileName, SchemaFile, HasHeader,
                                   ColumnDelimiter, textQualifier)
        elif IgnoreHeader == "True" and HasHeader == "True":
            dfFile = isIgnoreHeader(FullFileName, SchemaFile, ColumnDelimiter,
                                    textQualifier)
        
        # 2. Check if the file contains any rows of data   
        if len(dfFile.take(1)) > 0:
            filtered_cols = [col_name for col_name in dfFile.columns if not col_name.startswith("Filler_")]
            dfFile = dfFile.select(filtered_cols)

            dfFile = dfFile.withColumn("FILE_ID", lit(FileId)) \
                .withColumn("FILE_LAYOUT_ID", lit(FileLayoutId)) \
                .withColumn("FILE_LAYOUT_DESCRIPTION", lit(FileLayoutDescription)) \
                .withColumn("CLIENT_ID", lit(ClientId)) \
                .withColumn("LOAD_DATETIME", to_timestamp(current_timestamp(), "MM/dd/yyyy HH:mm:ss"))

            # 4. Append the processed data to the final storage location
            # Check if ProcessedPath is a table name (catalog.schema.table) or a file path
            if ProcessedPath.count('.') == 2 and not ProcessedPath.startswith('/'):
                # It's a table name - use Delta format (required for UC managed tables)
                dfFile.write.format("delta").mode("append").saveAsTable(ProcessedPath)
            else:
                # It's a file path - use Parquet format
                dfFile.write.format("parquet").mode("append").save(ProcessedPath)

            # 5. Log a successful processing report with the total row count
            rJSON.addNewEntry("Status", "SUCCESS")
            rJSON.addNewEntry("ProcessedCount", str(dfFile.count()))
            rJSON.addNewEntry("ProcessedCount", str(dfFile.count()))
            rJSON.addNewEntry("ErrorMessage", "", newLine=False)

        else:
            # Empty DataFrame after filtering - not an error, just no data
            rJSON.addNewEntry("Status", "SUCCESS")
            rJSON.addNewEntry("ProcessedCount", "0")
            rJSON.addNewEntry("ErrorMessage", "No records after filtering", newLine=False)

    except Exception as e:
        # 7. Catch any system crashes, clean the error text, and log a failed report
        clean_err = str(e).strip().replace(doubleQuote, "").replace("\n", " ").replace("\r", " ").replace("\t", " ")
        rJSON.addNewEntry("Status", "FAILED")
        rJSON.addNewEntry("ProcessedCount", "0")
        rJSON.addNewEntry("ErrorMessage", clean_err, newLine=False) 

    rJSON.addBraceEnd()
    return rJSON.getJSON()
